# Exercice 5 - Portefeuille homogène et non homogène (Vasicek)

In [ ]:
import numpy as np
from scipy.stats import norm, multivariate_normal
from scipy.integrate import quad
import matplotlib.pyplot as plt

## Partie A - Portefeuille homogène granulaire

### Question 1 : Moyenne et variance

Modele de Vasicek : défaut si $X_i = \sqrt{\rho} F + \sqrt{1-\rho} \varepsilon_i \leq s$ avec $s = \Phi^{-1}(PD)$.

Quand $N \to \infty$, par la LGN conditionnellement à F :
$$L | F = LGD \cdot \Phi\left(\frac{s - \sqrt{\rho} F}{\sqrt{1-\rho}}\right)$$

**Moyenne :** $E[L] = LGD \cdot PD$ (par proba totale, $E[p(F)] = PD$)

**Variance :** En passant à la limite granulaire :
$$Var(L) = LGD^2 \cdot \left[\Phi_2(s, s, \rho) - PD^2\right]$$

où $\Phi_2$ est la CDF de la loi normale bivariée avec correlation $\rho$.

In [ ]:
PD = 0.02
LGD = 0.45
rho = 0.15

In [ ]:
s = norm.ppf(PD)

# moyenne
EL = LGD * PD
print(f"E[L] = LGD * PD = {EL:.6f} = {EL*100:.3f}%")

# variance (via normale bivariée)
cov_mat = [[1, rho], [rho, 1]]
joint = multivariate_normal.cdf([s, s], mean=[0,0], cov=cov_mat)
var_L = LGD**2 * (joint - PD**2)
print(f"Var(L) = {var_L:.6f}")
print(f"Ecart-type = {np.sqrt(var_L)*100:.3f}%")

### Question 2 : VaR (Vasicek)

La formule fermée de Vasicek pour le quantile de perte :

$$VaR_\alpha = LGD \cdot \Phi\left(\frac{\Phi^{-1}(PD) + \sqrt{\rho}\, \Phi^{-1}(\alpha)}{\sqrt{1-\rho}}\right)$$

Ca vient de l'inversion de la relation $L|F$ : on cherche le F correspondant au quantile $\alpha$.

In [ ]:
def VaR_vasicek(alpha, PD, LGD, rho):
    return LGD * norm.cdf((norm.ppf(PD) + np.sqrt(rho)*norm.ppf(alpha)) / np.sqrt(1-rho))

In [ ]:
for alpha in [0.95, 0.99, 0.995, 0.999]:
    v = VaR_vasicek(alpha, PD, LGD, rho)
    print(f"VaR({alpha*100:.1f}%) = {v*100:.3f}%   Capital éco = {(v-EL)*100:.3f}%")

### Question 3 : $E[(L - l_0)^+]$

C'est l'expected loss de la tranche au dessus du seuil $l_0$. On intègre numeriquement :

In [ ]:
def EL_senior(l0, PD, LGD, rho):
    """E[(L-l0)+]"""
    def integrande(f):
        L_f = LGD * norm.cdf((norm.ppf(PD) - np.sqrt(rho)*f) / np.sqrt(1-rho))
        return max(L_f - l0, 0) * norm.pdf(f)
    result, _ = quad(integrande, -6, 6)
    return result

In [ ]:
l0_vals = np.linspace(0, LGD*0.8, 30)
el_vals = [EL_senior(l0, PD, LGD, rho) for l0 in l0_vals]

plt.figure(figsize=(8, 4))
plt.plot(l0_vals*100, [e*100 for e in el_vals], 'b-', lw=2)
plt.xlabel('Seuil $l_0$ (%)')
plt.ylabel('$E[(L-l_0)^+]$ (%)')
plt.title('EL tranche senior vs point d\'attachement')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Partie B - Portefeuille non homogène

Chaque actif a sa propre proba de défaut. Le rendement :
$$R_i = \rho F + \sqrt{1-\rho} \varepsilon_i \leq s + \sigma \varepsilon_i'$$

### Question 1 : Montrer que c'est encore un Vasicek

Défaut ssi : $\rho F + \sqrt{1-\rho} \varepsilon_i - \sigma \varepsilon_i' \leq s$

On pose $Z_i = \sqrt{1-\rho} \varepsilon_i - \sigma \varepsilon_i'$ qui est gaussien centré de variance $1-\rho + \sigma^2$.

La variance totale du membre de gauche est $\rho^2 + (1-\rho) + \sigma^2$. On pose $\sigma_{tot}^2 = \rho^2 + (1-\rho) + \sigma^2$.

En normalisant, on retrouve la structure Vasicek avec :
$$\rho^* = \frac{\rho^2}{\sigma_{tot}^2}, \quad s^* = \frac{s}{\sigma_{tot}}$$

Et $PD^* = \Phi(s^*)$.

In [ ]:
sigma = 0.20  # heterogeneité

sig_tot2 = rho**2 + (1-rho) + sigma**2
sig_tot = np.sqrt(sig_tot2)

rho_star = rho**2 / sig_tot2
s_star = s / sig_tot
PD_star = norm.cdf(s_star)

print(f"Parametres originaux : PD = {PD:.4f}, rho = {rho:.4f}")
print(f"Parametres modifiés : PD* = {PD_star:.4f}, rho* = {rho_star:.4f}")

In [ ]:
# comparaison des VaR
var_homogene = VaR_vasicek(0.999, PD, LGD, rho)
var_hetero = VaR_vasicek(0.999, PD_star, LGD, rho_star)

print(f"VaR 99.9% homogène    = {var_homogene*100:.3f}%")
print(f"VaR 99.9% hétérogène  = {var_hetero*100:.3f}%")
print(f"\nL'hétérogénéité réduit la corrélation effective et donc le risque de queue.")